# CNN β-microniches → guide enrichment

## Link to Zhang et al. Cell 2026 (SPAC-seq)

The paper shows that CRISPR KOs reshape **where** cells survive in tissue, not just **what** they express:

| Paper theme | Perturbation | Spatial/composition phenotype | Our proxy |
| --- | --- | --- | --- |
| **Immune exclusion** | sgIcam1 (lung M001) | sgIcam1+ tumor accumulates in immune-cold niches; IFN/LFA-1↓, T cells↓, M2/Spp1↑ | Predicted exclusion index + Icam1 CNN β; validate log₂ OR across CNN microniches |
| **Cd44–Spp1 crosstalk** | sgCd44 / sgSpp1 (lung); sgBcam in subQ | Macrophage Spp1 couples to T-cell Cd44; exhaustion/ECM programs | sgBcam in subQ/lung; Spp1/ECM escape module in predicted score |
| **Antigen presentation** | sgIl4ra, sgCd83, sgCd74 (subQ expanded) | MHC-II / costimulation down in immune niches | Immune-infiltration vs exclusion balance in niche score |

**This notebook:** each scatter point is a **tumor microniche** (CNN β-Leiden cluster). We ask whether niches SpaceTravLR scores as guide-favorable match niches where sgP cells are actually over-represented vs NTC.

Refresh data (higher niche resolution + spatial maps): set `REFRESH_CNN=True` below or run:
`python3 scripts/23_cnn_microniche_enrichment.py --leiden-resolution 0.9`


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "scripts" / "nb_common.py").exists():
    pass
elif (ROOT.parent / "scripts" / "nb_common.py").exists():
    ROOT = ROOT.parent
else:
    raise RuntimeError(
        "Start Jupyter from analysis/spacseq_tardis_validation/ "
        "(or open notebooks from that directory)."
    )

sys.path.insert(0, str(ROOT / "scripts"))
from nb_common import bootstrap, default_config, run_script, load_cache
import nb_viz

bootstrap()
CFG = default_config()
CACHE_TAG = CFG["tag"]
print("ROOT:", ROOT)
print("Config:", CFG)

from nb_common import run_script


In [ ]:
REFRESH_CNN = False
LEIDEN_RESOLUTION = 0.9  # higher → more microniches (default was 0.55)
if REFRESH_CNN:
    proc = run_script(
        "23_cnn_microniche_enrichment.py",
        "--tag", CFG["cnn_tag"],
        "--leiden-resolution", str(LEIDEN_RESOLUTION),
        "--min-ntc", "2", "--min-pert", "2",
    )
    print(proc.stdout[-3000:] if proc.stdout else proc.stderr)


In [ ]:
bundle = load_cache(CACHE_TAG)
print("Cached sections:", bundle.sections())
if bundle.missing():
    print(f"Warning: {len(bundle.missing())} artifacts missing from cache manifest")
bundle.missing()[:5]


In [ ]:
summary = bundle.json("cnn", "overall")
enrich = bundle.table("cnn", "enrichment")
corr = bundle.table("cnn", "corr")
print("Leiden resolution:", summary.get("leiden_resolution", "unknown"))
print("Median n niches:", corr["n_niches"].median())
summary, corr.sort_values("pearson_r", ascending=False).head(8)


In [ ]:
TOP_N = 6
POINT_COLOR = "#2563eb"
fig, axes = nb_viz.plot_cnn_enrichment_scatter(enrich, corr, top_n=TOP_N, tag=CFG["cnn_tag"], point_color=POINT_COLOR)
plt.show()


In [ ]:
fig, ax = nb_viz.plot_cnn_enrichment_heatmap(corr, tag=CFG["cnn_tag"], cmap="RdBu_r")
plt.show()


In [ ]:
# Spatial microniche map on tissue (cached parquet from script 23)
SPATIAL_SLICE = "Lung_Metastasis_M001"
SPATIAL_PERT = "Icam1"
spatial = bundle.spatial_tumor(SPATIAL_SLICE)
print("Available slices:", bundle.spatial_slices())
if spatial.empty:
    print("Run with REFRESH_CNN=True to generate spatial_tumor_*.parquet")
else:
    fig, axes = nb_viz.plot_microniche_spatial(
        spatial,
        slice_id=SPATIAL_SLICE,
        perturb=SPATIAL_PERT,
        panel="triple",
        point_size=3.5,
        title=f"{SPATIAL_SLICE} — paper Icam1 immune-exclusion niches (sg{SPATIAL_PERT})",
    )
    plt.show()


In [ ]:
# All tumor microniches on tissue (single panel)
if not spatial.empty:
    fig, ax = nb_viz.plot_microniche_spatial(
        spatial, slice_id=SPATIAL_SLICE, panel="all", point_size=4,
        title=f"{SPATIAL_SLICE} CNN β-microniches ({spatial['cnn_leiden'].nunique()} niches)",
    )
    plt.show()
